In [68]:
import pandas as pd
import numpy as np

from tqdm import tqdm_notebook
from scipy.spatial.distance import cityblock, cosine, euclidean, hamming, jaccard

from surprise import KNNWithMeans, KNNBaseline,  KNNBasic
from surprise import Dataset
from surprise import accuracy
from surprise import Reader
from surprise.model_selection import cross_validate
from surprise.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import fetch_openml



In [69]:
data = Dataset.load_builtin('ml-100k')

In [70]:
df = pd.DataFrame(data.raw_ratings, columns=['userId', 'movieId', 'rating', 'timestamp'])

In [71]:
df

,userId,movieId,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596
...,...,...,...,...
99995,880,476,3.0,880175444
99996,716,204,5.0,879795543
99997,276,1090,1.0,874795795
99998,13,225,2.0,882399156


In [72]:
#  посмотрим на размер датасета  (кол-во уникальных пользователей и фильмов)

print(f"фильмы: {len(df['movieId'].unique())}.")
print(f"пользователи: {len(df['userId'].unique())}.")

фильмы: 1682.
пользователи: 943.


In [73]:
# датасет для Surprise

dataset = pd.DataFrame( {
    'u_id'   : df['userId'],
    'i_id'   : df['movieId'],
    'rating' : df['rating']
})

dataset.head()

,u_id,i_id,rating
0,196,242,3.0
1,186,302,3.0
2,22,377,1.0
3,244,51,2.0
4,166,346,1.0


In [74]:
#  Max и Min значения по рейтингу
print( dataset['rating'].min(), dataset['rating'].max() )

1.0 5.0


In [75]:
# данные для создания модели

reader = Reader(rating_scale=(dataset['rating'].min(), dataset['rating'].max()))
data = Dataset.load_from_df(dataset, reader)

In [76]:
# создадим таблицу для записи результатов
# Метрики качества: | модель | подход | RMSE |
total = pd.DataFrame( columns=['модель','подход', 'RMSE'])
total

,модель,подход,RMSE


In [77]:
# модель KNNBaseline.
# user_based подход

algo = KNNBaseline(k=40, sim_options={'name': 'pearson_baseline', 'user_based': True})
res = cross_validate(algo, data, measures=['RMSE'], cv=50)

print(f"RMSE: {np.mean(res['test_rmse'])}.")
total = pd.concat( [total, pd.DataFrame([{ 'модель': 'KNNBaseline','подход': "user_based", 'RMSE': np.mean(res['test_rmse']) } ])\
                     ] )

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline si

In [78]:
# item based подход

algo = KNNBaseline(k=40, sim_options={'name': 'pearson_baseline', 'user_based': False})
res = cross_validate(algo, data, measures=['RMSE'], cv=50)

print(f"RMSE: {np.mean(res['test_rmse'])}.")
total = pd.concat( [total, pd.DataFrame([{ 'модель': 'KNNBaseline','подход': "item_based", 'RMSE': np.mean(res['test_rmse']) } ])\
                     ] )

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline si

In [79]:
total.sort_values(by = 'RMSE', ascending = True, inplace=True)

total

,модель,подход,RMSE
0,KNNBaseline,item_based,0.904087
0,KNNBaseline,user_based,0.909702


Лучшая модель KNNBaseline user based подход, RMSE = 0,91